## Sentinel

In [ ]:
import ee
import geemap


In [ ]:

# Initialize Earth Engine
ee.Initialize(project='satellite-segmentation-482111')

# Location (lat, lon)
lat = 12.9716
lon = 77.5946

# Point geometry
point = ee.Geometry.Point(lon, lat)

# Sentinel-2 Surface Reflectance
collection = (
    ee.ImageCollection("COPERNICUS/S2_SR")
    .filterBounds(point)
    .filterDate("2024-01-01", "2024-12-31")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
)

# Select least cloudy image
image = collection.sort("CLOUDY_PIXEL_PERCENTAGE").first()

# ---- TRUE COLOR (Natural Look) ----
true_color = image.select(["B4", "B3", "B2"])

# Visualization parameters (CRUCIAL)
vis_params = {
    "min": 0,
    "max": 3000,
    "gamma": 1.5
}


region = point.buffer(1000).bounds()

# Export to file
geemap.ee_export_image(
    #true_color,
    true_color.visualize(**vis_params),
    filename=r"F:\CDC\SEMANTIC SEGMENTATION - Copy\images_chk\new1.tif",
    region=region,
    scale=2,          # 10 m resolution
    file_per_band=False
)

print("✅ Satellite image downloaded (buildings, roads, vegetation visible)")


Generating URL ...
Please wait ...
Data downloaded to F:\CDC\SEMANTIC SEGMENTATION - Copy\images_chk\new1.tif
✅ Satellite image downloaded (buildings, roads, vegetation visible)


In [ ]:
from PIL import Image
import os

input_tif =test_img
output_png = r"F:\CDC\SEMANTIC SEGMENTATION - Copy\images_chk_png\new1.png"
img = Image.open(input_tif)

# Convert to RGB if needed
if img.mode != "RGB":
    img = img.convert("RGB")

img.save(output_png, format="PNG")

print('converted')
test_img_array= process_dataset(output_png,256)
print(test_img_array)

NameError: name 'test_img' is not defined

## Sentinel 2

In [ ]:
!pip install earthengine-api geemap


In [ ]:
import ee
import geemap

ee.Initialize(project='satellite-segmentation-482111')


In [ ]:
# Bangalore example
lon, lat = 77.5946, 12.9716

point = ee.Geometry.Point([lon, lat])
aoi   = point.buffer(5000).bounds()   # 5 km x 5 km AOI

start_date = "2024-01-01"
end_date   = "2024-12-31"


In [ ]:
def mask_s2_clouds(img):
    scl = img.select("SCL")

    # Keep pixels that are NOT:
    # 3 = cloud shadow
    # 8 = cloud medium probability
    # 9 = cloud high probability
    # 10 = thin cirrus
    # 11 = snow or ice
    mask = (
        scl.neq(3)
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    return (
        img.updateMask(mask)
           .divide(10000)   # scale reflectance
           .copyProperties(img, ["system:time_start"])
    )


s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi)
    .filterDate(start_date, end_date)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
    .map(mask_s2_clouds)
    .select(["B2", "B3", "B4", "B8"])
    .median()
)


In [ ]:
ndvi = s2.normalizedDifference(["B8", "B4"]).rename("NDVI")

# NDBI needs SWIR (B11 – 20 m, auto-resampled)
s2_swir = (
    ee.ImageCollection("COPERNICUS/S2_SR")
    .filterBounds(aoi)
    .filterDate(start_date, end_date)
    .map(mask_s2_clouds)
    .select(["B8", "B11"])
    .median()
)

ndbi = s2_swir.normalizedDifference(["B11", "B8"]).rename("NDBI")


In [ ]:
s1 = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(aoi)
    .filterDate(start_date, end_date)
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
    .select(["VV", "VH"])
    .median()
)


In [ ]:
final_stack = (
    s2
    .addBands(ndvi)
    .addBands(ndbi)
    .addBands(s1)
)

print("Final bands:", final_stack.bandNames().getInfo())


Final bands: ['B2', 'B3', 'B4', 'B8', 'NDVI', 'NDBI', 'VV', 'VH']


In [ ]:
aoi   = point.buffer(1000).bounds()

In [ ]:
geemap.ee_export_image(
    final_stack.select(["B4","B3","B2"]).visualize(
        min=0, max=0.3
    ),
    filename=r"F:\CDC\SEMANTIC SEGMENTATION - Copy\images_chk\new.tif",
    region=region,
    scale=2,
    file_per_band=False
)


Generating URL ...
Please wait ...
Data downloaded to F:\CDC\SEMANTIC SEGMENTATION - Copy\images_chk\new.tif


In [ ]:
Map = geemap.Map(center=[lat, lon], zoom=12)

Map.addLayer(
    final_stack.select(["B4","B3","B2"]),
    {"min": 0, "max": 0.3},
    "Sentinel-2 RGB"
)

Map.addLayer(ndvi, {"min": -1, "max": 1, "palette": ["brown","yellow","green"]}, "NDVI")
Map.addLayer(ndbi, {"min": -1, "max": 1, "palette": ["green","white","gray"]}, "NDBI")

Map


## mapbox


In [ ]:
import requests
import os

# Mapbox token
TOKEN = "YOUR_MAPBOX_TOKEN"

# Coordinates
lat, lon = 12.9716, 77.5946
zoom = 19

# Image size (max free = 1024x1024)
width, height = 1024, 1024

# Output folder
out_dir = r"C:\Users\CITERND\kishore-garvity\SEMANTIC SEGMENTATION\Semantic segmentation dataset"
os.makedirs(out_dir, exist_ok=True)

# API URL
url = (
    f"https://api.mapbox.com/styles/v1/mapbox/satellite-v9/static/"
    f"{lon},{lat},{zoom}/{width}x{height}"
    f"?access_token={TOKEN}"
)

# Download
response = requests.get(url)
response.raise_for_status()

# Save
output_path = os.path.join(out_dir, "mapbox_satellite.png")
with open(output_path, "wb") as f:
    f.write(response.content)

print("✅ High-resolution satellite image saved:", output_path)


## Sentinel hub

In [ ]:
!pip install sentinelhub

  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
Using cached dataclasses_json-0.6.7-py3-none-any.whl (28 kB)
Using cached typing_inspect-0.9.0-py3-none-any.whl (8.8 kB)
Using cached mypy_extensions-1.1.0-py3-none-any.whl (5.0 kB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)

   ----------------------------------------  0/10 [aenum]
   -------- -------------------------------  2/10 [tqdm]
   ------------------------ ---------------  6/10 [marshmallow]
   ------------------------------------ ---  9/10 [sentinelhub]
   ------------------------------------ ---  9/10 [sentinelhub]
   ---------------------------------------- 10/10 [sentinelhub]



In [ ]:
from sentinelhub import (
    SHConfig,
    BBox,
    CRS,
    SentinelHubRequest,
    DataCollection,
    MimeType,
    bbox_to_dimensions
)

ModuleNotFoundError: No module named 'sentinelhub'

In [ ]:


llat = 12.9716
lon = 77.5946

config = SHConfig()
config.sh_client_id = "8d4cf8da-7ad9-409f-a0f9-1090699928a6"
config.sh_client_secret = "ypAmlW9LISjWbt4MQkiCehbJryIiVLx6"
config.use_oauth = True

bbox = BBox(
    bbox=[lon - 0.01, lat - 0.01, lon + 0.01, lat + 0.01],
    crs=CRS.WGS84
)

resolution = 10  # meters per pixel
size = bbox_to_dimensions(bbox, resolution=resolution)

request = SentinelHubRequest(
    data_folder="output",
    evalscript="""
    //VERSION=3
    function setup() {
        return {
            input: ["B04", "B03", "B02"],
            output: { bands: 3 }
        };
    }
    function evaluatePixel(sample) {
        return [sample.B04, sample.B03, sample.B02];
    }
    """,
    input_data=[
        SentinelHubRequest.input_data(
            DataCollection.SENTINEL2_L2A,
            time_interval=("2024-01-01", "2024-12-30"),
        )
    ],
    responses=[SentinelHubRequest.output_response("default", MimeType.PNG)],
    bbox=bbox,
    size=(512, 512),
    config=config
)

data = request.get_data(save_data=True)
print("Download complete")


Download complete


## open street map

In [ ]:
%pip install osmnx geopandas shapely matplotlib networkx


In [ ]:
lat = 12.9716   # Bangalore
lon = 77.5946
radius = 1000   # meters (1 km area)

In [ ]:
import osmnx as ox

point = (lat, lon)


buildings = ox.features_from_point(
    point,
    tags={"building": True},
    dist=radius
)

print("Number of buildings:", len(buildings))


Number of buildings: 1889


##ESRI

In [ ]:
import requests
from PIL import Image
from io import BytesIO
import math
import os

def latlon_to_webmercator(lat, lon):
    """Convert lat/lon to EPSG:3857"""
    x = lon * 20037508.34 / 180
    y = math.log(math.tan((90 + lat) * math.pi / 360)) / (math.pi / 180)
    y = y * 20037508.34 / 180
    return x, y


def download_esri_satellite(lat, lon, zoom=18, size=512, save_path="esri.png"):

    if not save_path.lower().endswith(".png"):
        save_path += ".png"

    x, y = latlon_to_webmercator(lat, lon)

    meters_per_pixel = 156543.03392 / (2 ** zoom)
    half_size = (size / 2) * meters_per_pixel

    bbox = f"{x-half_size},{y-half_size},{x+half_size},{y+half_size}"

    url = "https://services.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/export"

    params = {
        "bbox": bbox,
        "bboxSR": 3857,
        "imageSR": 3857,
        "size": f"{size},{size}",
        "format": "png",
        "f": "image"
    }

    response = requests.get(url, params=params)

    # 🔍 DEBUG: Check if ESRI returned an error
    content_type = response.headers.get("Content-Type", "")
    if "image" not in content_type:
        print("❌ ESRI Error Response:")
        print(response.text[:500])
        return

    image = Image.open(BytesIO(response.content))
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    image.save(save_path)

    print(f"✅ Saved ESRI satellite image → {save_path}")


In [ ]:
download_esri_satellite(
    lat=47.4362,
    lon=-122.187,
    zoom=18,
    size=512,
    save_path="F:\CDC\SEMANTIC SEGMENTATION - Copy\images_chk_png\esri_3.png"
)

✅ Saved ESRI satellite image → F:\CDC\SEMANTIC SEGMENTATION - Copy\images_chk_png\esri_3.png


##modified esri

In [ ]:
import math
import requests
from PIL import Image
from io import BytesIO
import os

def latlon_to_tile(lat, lon, zoom):
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x_tile = int((lon + 180.0) / 360.0 * n)
    y_tile = int((1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return x_tile, y_tile


In [ ]:
def download_esri_patch(lat, lon, zoom=19, tiles=2, save_path="patch.png"):
    x0, y0 = latlon_to_tile(lat, lon, zoom)

    images = []
    for dy in range(tiles):
        row = []
        for dx in range(tiles):
            url = f"https://services.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{zoom}/{y0+dy}/{x0+dx}"
            img = Image.open(BytesIO(requests.get(url).content))
            row.append(img)
        images.append(row)

    tile_size = images[0][0].size[0]
    patch = Image.new("RGB", (tile_size*tiles, tile_size*tiles))

    for y in range(tiles):
        for x in range(tiles):
            patch.paste(images[y][x], (x*tile_size, y*tile_size))

    patch.save(save_path)
    print(f"✅ Saved SHARP patch → {save_path}")


In [ ]:
download_esri_patch(
    lat=47.2704,
    lon=-122.313,
    zoom=19,
    save_path=r"F:\CDC\SEMANTIC SEGMENTATION - Copy\images_chk_png\esri_sharp.png"
)


✅ Saved SHARP patch → F:\CDC\SEMANTIC SEGMENTATION - Copy\images_chk_png\esri_sharp.png


##Final esri

In [ ]:
import pandas as pd
import math
import requests
from PIL import Image
from io import BytesIO
import os
import time


In [ ]:
def latlon_to_tile(lat, lon, zoom):
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x_tile = int((lon + 180.0) / 360.0 * n)
    y_tile = int(
        (1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi)
        / 2.0 * n
    )
    return x_tile, y_tile


In [ ]:
def download_esri_tile_256(lat, lon, zoom, save_path):
    x, y = latlon_to_tile(lat, lon, zoom)

    url = f"https://services.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{zoom}/{y}/{x}"

    response = requests.get(url, timeout=10)

    if response.status_code != 200:
        print(f"❌ Failed tile @ {lat},{lon}")
        return False

    image = Image.open(BytesIO(response.content))

    if image.size != (256, 256):
        image = image.resize((256, 256), Image.BILINEAR)

    image.save(save_path)
    return True


In [ ]:
def download_from_csv(
    csv_path,
    out_dir,
    zoom=19,
    start_index=0
):
    df = pd.read_csv(csv_path)

    os.makedirs(out_dir, exist_ok=True)

    for idx, row in df.iterrows():
        lat = row["lat"]
        lon = row["lon"]

        filename = f"img_{idx+start_index:05d}.png"
        save_path = os.path.join(out_dir, filename)

        success = download_esri_tile_256(
            lat, lon,
            zoom=zoom,
            save_path=save_path
        )

        if success:
            print(f"✅ Saved {filename}")
        else:
            print(f"⚠️ Skipped {filename}")

        time.sleep(0.25)  # 🛑 rate-limit (important)


In [ ]:
download_from_csv(
    csv_path=r"F:\CDC\SEMANTIC SEGMENTATION - Copy\training_tabular.csv",
    out_dir=r"F:\CDC\SEMANTIC SEGMENTATION - Copy\satellite_images",
    zoom=19
)


## taken from web

In [ ]:
import math
import time
import requests
from PIL import Image
from io import BytesIO

ZOOM = 19
TILE_SIZE = 256
TILES_AROUND = 1  # 3x3 grid

HEADERS = {
    "User-Agent": "HousePriceMultimodalProject/1.0 (educational)"
}

def latlon_to_tile(lat, lon, zoom):
    lat_rad = math.radians(lat)
    n = 2.0 ** zoom
    xtile = int((lon + 180.0) / 360.0 * n)
    ytile = int(
        (1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi)
        / 2.0 * n
    )
    return xtile, ytile


def fetch_satellite_image(lat, lon, save_path):
    x, y = latlon_to_tile(lat, lon, ZOOM)

    img_size = TILE_SIZE * (2 * TILES_AROUND + 1)
    final_img = Image.new("RGB", (img_size, img_size))

    for dx in range(-TILES_AROUND, TILES_AROUND + 1):
        for dy in range(-TILES_AROUND, TILES_AROUND + 1):
            tile_x = x + dx
            tile_y = y + dy

            url = (
                f"https://services.arcgisonline.com/ArcGIS/rest/services/"
                f"World_Imagery/MapServer/tile/{ZOOM}/{tile_y}/{tile_x}"
            )

            r = requests.get(url, headers=HEADERS, timeout=10)
            tile_img = Image.open(BytesIO(r.content)).convert("RGB")

            px = (dx + TILES_AROUND) * TILE_SIZE
            py = (dy + TILES_AROUND) * TILE_SIZE
            final_img.paste(tile_img, (px, py))

    final_img.save(save_path)

In [ ]:
lat=47.2704
lon=-122.313

In [ ]:
fetch_satellite_image(lat, lon,r"F:\CDC\SEMANTIC SEGMENTATION - Copy\images_chk_png\new1.png" )